In [ ]:
# === GOOGLE COLAB SETUP ===
import os
if 'COLAB_GPU' in os.environ:
    print("Running on Google Colab. Setting up environment...")
    os.chdir('/content')
    if not os.path.exists('btc_ai'):
        !git clone https://github.com/taybro-o/btc_ai.git
    os.chdir('/content/btc_ai')
    !pip install -q scikit-learn matplotlib pandas numpy tensorflow requests
    print("Environment ready!")
else:
    print("Running locally. Skipping Colab setup.")

# BTC AI Master Notebook (Overfitting & Data Scale Optimization)
Includes Historical Backfill (20,000 candles) and increased Regularization.

In [ ]:
import os, time, requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler

DATA_DIR = "data"
MASTER_FILE = os.path.join(DATA_DIR, "btcusdt_analysis_data.csv")

def fetch_klines(symbol="BTCUSDT", interval="1m", limit=1000, start_time=None):
    url = f"https://api.binance.com/api/v3/klines?symbol={symbol}&interval={interval}&limit={limit}"
    if start_time: url += f"&startTime={start_time}"
    res = requests.get(url, timeout=10)
    if res.status_code == 200:
        df = pd.DataFrame(res.json(), columns=['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6'])
        df[['Open', 'High', 'Low', 'Close', 'Volume']] = df[['Open', 'High', 'Low', 'Close', 'Volume']].apply(pd.to_numeric)
        return df[['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume']]
    return None

def update_and_prune_data(target_rows=20000):
    if not os.path.exists(DATA_DIR): os.makedirs(DATA_DIR, exist_ok=True)
    master_df = pd.read_csv(MASTER_FILE) if os.path.exists(MASTER_FILE) else pd.DataFrame()
    
    # Backfill if dataset is too small
    while len(master_df) < target_rows:
        print(f"Dataset small ({len(master_df)} rows). Backfilling...")
        end_ts = master_df['Timestamp'].min() if not master_df.empty else int(time.time() * 1000)
        # Fetch 1000 minutes before the current oldest timestamp
        start_ts = end_ts - (1000 * 60 * 1000)
        new_df = fetch_klines(start_time=start_ts)
        if new_df is None or new_df.empty: break
        master_df = pd.concat([new_df, master_df]).drop_duplicates(subset=['Timestamp']).sort_values('Timestamp')
        time.sleep(0.5) # Rate limit respect
    
    # Always fetch latest
    latest_df = fetch_klines()
    master_df = pd.concat([master_df, latest_df]).drop_duplicates(subset=['Timestamp']).sort_values('Timestamp')
    
    master_df = master_df.tail(target_rows)
    master_df.to_csv(MASTER_FILE, index=False)
    return master_df

df = update_and_prune_data()
df['Datetime'] = pd.to_datetime(df['Timestamp'], unit='ms')
df.set_index('Datetime', inplace=True)

def apply_indicators(df):
    df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))
    df['EMA_50'] = df['Close'].ewm(span=50).mean()
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + (gain / (loss + 1e-9))))
    df['MACD'] = df['Close'].ewm(span=12).mean() - df['Close'].ewm(span=26).mean()
    df['dist_from_ema'] = (df['Close'] - df['EMA_50']) / df['EMA_50']
    df['smoothed_return'] = df['log_return'].rolling(window=5).mean()
    df.dropna(inplace=True)
    return df

df = apply_indicators(df)

LOOKBACK = 256
FORECAST = 5
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'EMA_50', 'RSI', 'MACD', 'dist_from_ema', 'log_return', 'smoothed_return']

split_idx = int(len(df) * 0.9)
scaler = StandardScaler()
scaler.fit(df.iloc[:split_idx][features])
scaled_data = scaler.transform(df[features])

def create_sequences(data, lookback, forecast):
    X, y = [], []
    for i in range(len(data) - lookback - forecast + 1):
        X.append(data[i : i + lookback])
        y.append(data[i + lookback : i + lookback + forecast, -2])
    return np.array(X, dtype='float32'), np.array(y, dtype='float32')

X_all, y_all = create_sequences(scaled_data, LOOKBACK, FORECAST)
split = int(len(X_all) * 0.9)
X_train, X_val = X_all[:split], X_all[split:]
y_train, y_val = y_all[:split], y_all[split:]

print(f"Data Prepared. Train: {X_train.shape} samples (~{X_train.shape[0]//128} steps/epoch), Val: {X_val.shape}")

In [ ]:
def directional_accuracy(y_true, y_pred):
    return tf.reduce_mean(tf.cast(tf.equal(tf.sign(y_true), tf.sign(y_pred)), tf.float32))

def custom_directional_loss(y_true, y_pred):
    mse = tf.math.square(y_true - y_pred)
    # Aggressive penalty for sign mismatch
    direction_penalty = 15.0 * tf.maximum(0.0, -y_true * y_pred)
    return tf.reduce_mean(mse + direction_penalty)

def create_patch_tst(lookback, num_channels, patch_len, model_dim, num_heads, num_layers, forecast_len):
    inputs = layers.Input(shape=(lookback, num_channels))
    
    # 1. RevIN
    mu = layers.Lambda(lambda x: tf.reduce_mean(x, axis=1, keepdims=True))(inputs)
    std = layers.Lambda(lambda x: tf.math.reduce_std(x, axis=1, keepdims=True) + 1e-5)(inputs)
    x = layers.Lambda(lambda args: (args[0] - args[1]) / args[2])([inputs, mu, std])
    
    # 2. Patching
    x = layers.Permute((2, 1))(x)
    num_patches = lookback // patch_len
    x = layers.Reshape((num_channels, num_patches, patch_len))(x)
    x = layers.Dense(model_dim)(x)
    
    # 3. Positional Embedding
    class PositionalEmbedding(layers.Layer):
        def __init__(self, **kwargs):
            super().__init__(**kwargs)
            self.pos_embed = self.add_weight(
                name='pos_enc', shape=(1, 1, num_patches, model_dim),
                initializer='random_normal', trainable=True
            )
        def call(self, x): return x + self.pos_embed
    x = PositionalEmbedding()(x)
    
    # 4. Transformer Backbone (High Dropout to prevent overfitting)
    for _ in range(num_layers):
        attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=model_dim//num_heads)(x, x)
        x = layers.LayerNormalization(epsilon=1e-6)(layers.Add()([x, layers.Dropout(0.3)(attn)]))
        ff = layers.Dense(model_dim*4, activation='gelu')(x)
        ff = layers.Dropout(0.3)(ff)
        ff = layers.Dense(model_dim)(ff)
        x = layers.LayerNormalization(epsilon=1e-6)(layers.Add()([x, layers.Dropout(0.3)(ff)]))
    
    # 5. Prediction Head
    x = layers.Reshape((num_channels, num_patches * model_dim))(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(forecast_len)(x)
    
    # 6. Extract Target and Denormalize
    t_mu, t_std = mu[:, 0, -2:-1], std[:, 0, -2:-1]
    t_pred = x[:, -2, :]
    outputs = layers.Lambda(lambda args: args[0] * args[1] + args[2])([t_pred, t_std, t_mu])
    
    return tf.keras.Model(inputs=inputs, outputs=outputs)

model = create_patch_tst(LOOKBACK, len(features), 16, 32, 4, 3, FORECAST)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=custom_directional_loss, metrics=['mae', directional_accuracy])

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, batch_size=128, callbacks=callbacks)

In [ ]:
predictions = model.predict(X_val)
ret_mean, ret_std = scaler.mean_[-2], scaler.scale_[-2]
pred_returns = (predictions * ret_std) + ret_mean
last_price = df['Close'].iloc[-1]
last_timestamp = df.index[-1]

print('\n--- 5-MINUTE REAL-TIME FORECAST ---')
print(f'Market Data as of: {last_timestamp} UTC')
print('------------------------------------------')
initial_price = last_price
for i in range(5):
    forecast_time = last_timestamp + timedelta(minutes=i+1)
    forecasted_return = pred_returns[-1, i]
    p = last_price * np.exp(forecasted_return)
    direction = 'UP' if forecasted_return > 0 else 'DOWN'
    pct_change = ((p / initial_price) - 1) * 100
    print(f"{forecast_time.strftime('%H:%M')} | {direction} | Target: {p:.2f} USD | Total: {pct_change:+.4f}%")
    last_price = p